Step 1: Cloning the Git Repo

In [ ]:
import git
import os

# Function to clone a Git repository
def clone_repo(repo_url, base_path="repos"):
    repo_name = repo_url.split("/")[-1].replace(".git", "")
    repo_path = os.path.join(base_path, repo_name)

    if not os.path.exists(repo_path):
        git.Repo.clone_from(repo_url, repo_path)
    return repo_path


In [ ]:
IGNORE_DIRS = {"__pycache__", ".git", "venv", "env", "node_modules"}
VALID_EXTENSIONS = {".py", ".js", ".java", ".cpp", ".c", ".h", ".go", ".rb", ".php"}

def scan_repo(repo_path):
    files = []

    for root, dirs, filenames in os.walk(repo_path):
        # Skip ignored directories
        dirs[:] = [d for d in dirs if d not in IGNORE_DIRS]

        for filename in filenames:
            if any(filename.endswith(ext) for ext in VALID_EXTENSIONS):
                file_path = os.path.join(root, filename)
                files.append(file_path)

    return files

Step 2: Chunking

In [ ]:
def chunk_code(code, max_chars=3000):
    return [code[i:i + max_chars] for i in range(0, len(code), max_chars)]

Step 3: Parsing(AST-based)

In [ ]:
import ast

def parse_code(file_path):
    with open(file_path, "r") as f:
        tree = ast.parse(f.read())
    
    functions = []
    classes = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            functions.append(node.name)
        elif isinstance(node, ast.ClassDef):
            classes.append(node.name)
    return {
        "functions": functions,
        "classes": classes
    }

Step 4: LLM Analyzer

In [ ]:
def analyze_file(file_path):
    with open(file_path, "r") as f:
        code = f.read()
    
    chunks = chunk_code(code)
    analyses = []
    for chunk in chunks:
        prompt = FILE_ANALYSIS_PROMPT.format(code=chunk)
        response = llm.invoke(prompt)
        analyses.append(response)
    
    return "\n".join(analyses)

Step 5: Intermediate Summarization

In [ ]:
def summarize(llm, analysis_text):
    prompt = f"""
    Summarize the following code analysis into concise developer-friendly notes:
    {analysis_text}
    """
    return llm.invoke(prompt)

Step 6: Generating Document

In [ ]:
def generate_docs(llm, file_path, summary):
    prompt = f"""
    Generate Markdown documentation for this file.

    Include:
    - Purpose
    - Key Functions
    - Data Flow
    - Important Notes

    File: {file_path}

    Summary:
    {summary}
    """
    return llm.invoke(prompt)

In [ ]:
def generate_folder_readme(llm, folder_path, file_docs):
    combined = "\n\n".join(file_docs)
    prompt = f"""
    Generate a README.md for this folder.

    Include:
    - Overview
    - Key files
    - How components interact

    Content:
    {combined}
    """
    return llm.invoke(prompt)

In [ ]:
def pipeline(repo_url, llm):
    repo_path = clone_repo(repo_url)
    files = scan_repo(repo_path)

    file_docs = []
    summaries = []
    for file_path in files:
        try:
            analysis = analyze_file(file_path)
            summary = summarize(llm, analysis)
            summaries.append(summary)
            doc = generate_docs(llm, file_path, summary)
            file_docs.append(doc)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    readme_content = generate_folder_readme(llm, repo_path, file_docs)
    with open(os.path.join(repo_path, "README.md"), "w") as f:
        f.write(readme_content)

In [9]:
%pip install langchain langchain-openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install openai tiktoken python-dotenv

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [3]:
import os
from langchain_openai import OpenAI

llm = OpenAI(model="gpt-4o-mini", temperature=0.2)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# repo_url = "https://github.com/Drag0nop/ai-task-remainder-agent"